# Ollama Colab

Run a large language model on a free Google Colab GPU and expose it as a public API endpoint.

| Feature | Details |
|---|---|
| Model selection | Browse & search the live Ollama library — pick any model |
| Tunnel options | Cloudflare quick tunnel *(default)*, Cloudflare named tunnel, ngrok |

**Setup**: `Runtime → Change runtime type → T4 GPU → Save`, then run cells in order.

## Step 1: Install Dependencies

In [ ]:
# ── Model Discovery with Robust Fallbacks ───────────────────────────────────────────────
import requests
from bs4 import BeautifulSoup

# Static fallback list of popular models (updated 2025)
STATIC_FALLBACK_MODELS = [
    {"name": "qwen2.5", "desc": "General purpose language model", "pulls": "High", "tags": ["14b", "7b", "32b"]},
    {"name": "qwen2.5-coder", "desc": "Code-specialized model", "pulls": "High", "tags": ["14b", "7b", "32b"]},
    {"name": "deepseek-r1", "desc": "Reasoning model with chain-of-thought", "pulls": "Very High", "tags": ["14b", "32b", "70b"]},
    {"name": "llama3.1", "desc": "Meta's open language model", "pulls": "Very High", "tags": ["8b", "70b"]},
    {"name": "llama3.2", "desc": "Latest Meta model with improved performance", "pulls": "High", "tags": ["1b", "3b", "8b"]},
    {"name": "mistral", "desc": "Efficient general purpose model", "pulls": "High", "tags": ["7b"]},
    {"name": "codellama", "desc": "Code generation specialist", "pulls": "Medium", "tags": ["13b", "7b", "34b"]},
    {"name": "deepseek-coder", "desc": "Code-specialized model", "pulls": "Medium", "tags": ["6.7b", "33b"]},
    {"name": "gemma2", "desc": "Google's open model family", "pulls": "Medium", "tags": ["9b", "27b"]},
    {"name": "phi3", "desc": "Microsoft's small but capable model", "pulls": "Medium", "tags": ["3.8b", "14b"]},
]

def fetch_ollama_library():
    """Fetch models from ollama.com with multiple fallback strategies"""
    
    # Strategy 1: Try web scraping with multiple selectors
    try:
        print("Attempting to fetch from ollama.com/library ...")
        resp = requests.get("https://ollama.com/library", timeout=15,
                          headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        
        models = []
        # Try multiple selector patterns for robustness
        selectors = [
            # Pattern 1: Current (2025) structure
            "a[href^='/library/']",
            # Pattern 2: Legacy structure
            "ul li a[href^='/library/']",
            # Pattern 3: Generic fallback
            "a[href*='/library/']"
        ]
        
        for selector in selectors:
            try:
                links = soup.select(selector)
                if links and len(links) > 10:  # Reasonable number of models found
                    print(f"✅ Using selector: {selector}")
                    for a in links:
                        href = a.get("href", "")
                        if href.startswith("/library/") and href.count("/") == 2:
                            name = href.split("/")[-1]
                            if any(m["name"] == name for m in models): 
                                continue
                            
                            # Try multiple element selectors for name/desc
                            name_el = a.select_one("h2, [class*='truncate'], [class*='name'], h3")
                            desc_el = a.select_one("p, [class*='description'], [class*='subtitle']")
                            pulls_el = a.find(string=lambda t: t and "Pull" in t)
                            
                            desc = desc_el.get_text(strip=True) if desc_el else ""
                            pulls = pulls_el.strip() if pulls_el else ""
                            
                            # Extract tags from various elements
                            tags = []
                            for tag_el in a.select("span, li, [class*='tag'], [class*='size']"):
                                tag_text = tag_el.get_text(strip=True)
                                if tag_text and len(tag_text) < 20:
                                    tags.append(tag_text)
                            
                            # Clean and deduplicate tags
                            seen = set()
                            unique_tags = []
                            for t in tags:
                                if t not in seen and t.lower() != name.lower():
                                    seen.add(t)
                                    unique_tags.append(t)
                            
                            models.append({
                                "name": name,
                                "desc": desc,
                                "pulls": pulls,
                                "tags": unique_tags,
                            })
                    
                    if models:
                        print(f"✅ Successfully scraped {len(models)} models")
                        return models
                        
            except Exception as e:
                print(f"⚠️ Selector {selector} failed: {e}")
                continue
                
    except Exception as e:
        print(f"⚠️ Web scraping failed completely: {e}")
    
    # Strategy 2: Try Ollama API (if available)
    try:
        print("🔄 Trying Ollama API fallback...")
        # Note: This would require Ollama server running, so it's a secondary fallback
        # For now, we'll skip this as it's not practical in Colab setup
        pass
    except Exception as e:
        print(f"⚠️ API fallback failed: {e}")
    
    # Strategy 3: Use static fallback list
    print("🔄 Using static fallback model list...")
    print(f"⚠️ Note: Using {len(STATIC_FALLBACK_MODELS)} popular models only.")
    print("💡 Full model browser will work when ollama.com scraping is fixed.")
    
    return STATIC_FALLBACK_MODELS

def fetch_tags(model_name):
    """Fetch tags with robust error handling"""
    if model_name in _tag_cache:
        return _tag_cache[model_name]
    
    try:
        print(f"Fetching tags for {model_name}...")
        resp = requests.get(f"https://ollama.com/library/{model_name}/tags", 
                          timeout=10,
                          headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        
        tags = []
        # Try multiple patterns for tag extraction
        tag_selectors = [
            "a[href*=':']",
            "span[class*='tag']", 
            "li",
            "[class*='version']"
        ]
        
        for selector in tag_selectors:
            try:
                elements = soup.select(selector)
                for el in elements:
                    href = el.get("href", "")
                    text = el.get_text(strip=True)
                    
                    # Extract tag from href or text
                    if ":" in href:
                        tag = href.split(":")[-1].split("/")[0].strip()
                    elif text and len(text) < 20 and not text.startswith("http"):
                        tag = text
                    else:
                        continue
                        
                    if tag and ":" not in tag and tag not in tags:
                        tags.append(tag)
                
                if tags:
                    print(f"✅ Found {len(tags)} tags for {model_name}")
                    break
                    
            except Exception as e:
                print(f"⚠️ Tag selector {selector} failed: {e}")
                continue
        
        # If no tags found, provide sensible defaults
        if not tags:
            print(f"⚠️ No tags found for {model_name}, using defaults")
            # Provide common tag variations based on model name
            if "coder" in model_name.lower():
                tags = ["14b", "7b", "32b"]
            elif "r1" in model_name.lower():
                tags = ["14b", "32b", "70b"]
            elif "llama" in model_name.lower():
                tags = ["8b", "70b"]
            else:
                tags = ["14b", "7b"]
        
        _tag_cache[model_name] = tags
        return tags
        
    except Exception as e:
        print(f"⚠️ Tag fetch failed for {model_name}: {e}")
        # Provide fallback tags
        fallback_tags = ["latest"]
        if "coder" in model_name.lower():
            fallback_tags = ["14b", "latest"]
        elif "r1" in model_name.lower():
            fallback_tags = ["14b", "latest"]
            
        _tag_cache[model_name] = fallback_tags
        return fallback_tags

# Initialize tag cache
_tag_cache = {}

# ── Size and Capability Detection ───────────────────────────────────────────────
SIZE_SUFFIXES = ("b", "m", "k")

def size_tags(model):
    return [t for t in model["tags"]
            if t.lower().replace(".","").rstrip("bmk").isdigit() or
               any(t.lower().endswith(s) for s in SIZE_SUFFIXES)]

def model_matches_cap(model, cap):
    if cap == "all":
        return True
    kws = CAP_KEYWORDS[cap]
    combined = (model["name"] + " " + " ".join(model["tags"])).lower()
    return any(k in combined for k in kws)

# ── Capability categories ─────────────────────────────────────────────────────
CAPABILITIES = ["all", "reasoning", "tools", "vision", "embedding", "code"]
CAP_KEYWORDS  = {
    "reasoning": ["reasoning", "r1", "thinking"],
    "tools":     ["tools"],
    "vision":    ["vision"],
    "embedding": ["embedding", "embed"],
    "code":      ["code", "coder", "coding", "starcoder", "deepseek-coder", "codellama"],
    "cloud":     ["cloud"],
}

# ── Widgets ──────────────────────────────────────────────────────────────────
_style  = {"description_width": "80px"}
_layout = widgets.Layout(width="100%")

w_search = widgets.Text(
    placeholder="Search models...", description="Search:", style=_style, layout=_layout)
w_cap = widgets.ToggleButtons(
    options=CAPABILITIES, value="all", description="Filter:",
    style={"button_width": "90px", "description_width": "80px"},
    layout=_layout)
w_list = widgets.Select(
    options=[], rows=12, description="Model:", style=_style,
    layout=widgets.Layout(width="100%"))
w_tag_label = widgets.HTML("<b>Tag / variant:</b>")
w_tags = widgets.RadioButtons(options=[], description="", layout=_layout)
w_info  = widgets.HTML("")
w_apply = widgets.Button(description="✅  Use this model",
                         button_style="success",
                         layout=widgets.Layout(width="220px"))
w_out   = widgets.Output()

# ── Refresh model list ────────────────────────────────────────────────────────
def refresh_list(*_):
    query = w_search.value.strip().lower()
    cap   = w_cap.value
    filtered = [m for m in ALL_MODELS
                if model_matches_cap(m, cap)
                and (not query or query in m["name"].lower() or query in m["desc"].lower())]
    labels = []
    for m in filtered:
        sizes = [t for t in size_tags(m)][:4]
        size_str = "  " + " ".join(sizes) if sizes else ""
        pulls = m["pulls"].replace(" Pulls", "").strip() if m["pulls"] else ""
        pulls_str = f"  [{pulls}]" if pulls else ""
        labels.append(f"{m['name']}{size_str}{pulls_str}")
    w_list.options = labels
    if labels:
        w_list.value = labels[0]

# ── Update tag radio buttons when model selected ──────────────────────────────
def on_model_select(change):
    if not w_list.value:
        return
    model_name = w_list.value.split()[0]
    w_info.value = "<i>Loading tags...</i>"
    w_tags.options = []
    tags = fetch_tags(model_name)
    ordered = sorted(tags, key=lambda t: (t != "latest", t))[:20]
    w_tags.options = ordered
    if ordered:
        w_tags.value = ordered[0]
    meta = next((m for m in ALL_MODELS if m["name"] == model_name), None)
    desc = meta["desc"] if meta else ""
    pulls = meta["pulls"] if meta else ""
    w_info.value = (
        f"<b>{model_name}</b>"
        + (f" — {desc}" if desc else "")
        + (f"<br><small>{pulls}</small>" if pulls else "")
    )

# ── Apply button ──────────────────────────────────────────────────────────────
MODEL_NAME = None

def on_apply(_):
    global MODEL_NAME
    if not w_list.value or not w_tags.value:
        return
    model_base = w_list.value.split()[0]
    tag        = w_tags.value
    MODEL_NAME = f"{model_base}:{tag}"
    with w_out:
        clear_output()
        print(f"✅  MODEL_NAME set to: {MODEL_NAME}")
        print("Continue to Step 3 when ready.")

w_search.observe(refresh_list, names="value")
w_cap.observe(refresh_list, names="value")
w_list.observe(on_model_select, names="value")
w_apply.on_click(on_apply)

# ── Initial render ────────────────────────────────────────────────────────────
print("🔄 Initializing model browser...")
ALL_MODELS = fetch_ollama_library()
refresh_list()

display(widgets.VBox([
    widgets.HTML("<h4>🦙 Ollama Model Browser</h4>"),
    widgets.HTML("<small><i>Note: If web scraping fails, using popular models list. Full browser available when ollama.com is accessible.</i></small><br><br>"),
    w_search,
    w_cap,
    w_list,
    w_tag_label,
    w_tags,
    w_info,
    w_apply,
    w_out,
], layout=widgets.Layout(padding="8px", border="1px solid #ccc", border_radius="6px")))

## Step 4: Ollama 2026 Features

The notebook supports the latest Ollama capabilities (v0.14.0+). These features are available through the standard Ollama HTTP API endpoints.

### 🚀 Anthropic API Compatibility (v0.14.0+)

Ollama now provides Anthropic-compatible API endpoints, enabling Claude Code integration and Anthropic-style API clients.

**Available Endpoints:**
- `/v1/messages` - Anthropic-style message API
- `/v1/models` - List available models

**Example Usage:**

```python
# After the server is running, use the Anthropic-compatible endpoint
import requests

# Replace with your tunnel URL
OLLAMA_URL = "<YOUR_TUNNEL_URL>"

# Anthropic-style message request
response = requests.post(f"{OLLAMA_URL}/v1/messages", json={
    "model": MODEL_NAME,
    "max_tokens": 1024,
    "messages": [
        {"role": "user", "content": "Hello! Can you help me with Python code?"}
    ]
})

print(response.json())
```

**Benefits:**
- Compatible with Claude Code and other Anthropic clients
- Standard message format with role-based conversations
- Streaming responses supported

---

### 📊 Structured Outputs (JSON Schema)

Generate valid JSON responses with schema validation for API consumers.

**Example Usage:**

```python
# JSON schema for structured output
schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "age": {"type": "number"},
        "skills": {
            "type": "array",
            "items": {"type": "string"}
        }
    },
    "required": ["name", "age"]
}

# Request with format parameter
response = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": MODEL_NAME,
    "prompt": "Generate a developer profile",
    "format": schema,
    "stream": False
})

result = response.json()
print(result["response"])  # Validated JSON output
```

**Use Cases:**
- API integration with strict data contracts
- Automated data extraction
- Configuration generation
- Structured data processing

---

### 🔍 Web Search Plugin

Enable web search capabilities via Ollama integrations (requires additional setup).

**Setup Instructions:**

```bash
# After Ollama is installed, enable web search
# This requires additional configuration outside the notebook

# Example: Configure web search integration
# (Note: This feature requires additional setup and credentials)
```

**Usage Example:**

```python
# Web search enabled request
response = requests.post(f"{OLLAMA_URL}/api/generate", json={
    "model": MODEL_NAME,
    "prompt": "What are the latest developments in AI in 2026?",
    "tools": ["web_search"],
    "stream": False
})

print(response.json()["response"])
```

**Requirements:**
- Additional Ollama configuration
- Search API credentials (e.g., Google Search API, Bing API)
- Network access from Colab environment

**Note:** Web search plugin requires manual setup and configuration. See [Ollama documentation](https://ollama.com/blog/web-search) for details.

---

### 📚 Feature Summary

| Feature | Version | Status | Use Case |
| --- | --- | --- | --- |
| Anthropic API | v0.14.0+ | ✅ Available | Claude Code integration |
| Structured Outputs | v0.14.0+ | ✅ Available | JSON schema validation |
| Web Search Plugin | v0.14.0+ | ⚠️ Manual Setup | Real-time information access |

These features are automatically available through the Ollama API endpoints exposed by the tunnel. No additional configuration is required for Anthropic API and Structured Outputs.

## Step 3: Configure Environment & Memory Management

### 🎛️ Memory Management

Choose how to handle model memory based on your needs and Colab's VRAM limits:

```python
# Memory Management Options
MEMORY_OPTIONS = {
    "keep_loaded": {
        "keep_alive": -1,  # Keep model in VRAM forever (current behavior)
        "description": "Fastest response times, but may cause OOM with large models"
    },
    "auto_unload": {
        "keep_alive": 0,   # Unload model immediately after each request
        "description": "Safe for any model size, but slower response times"
    },
    "5min": {
        "keep_alive": 300,  # Keep loaded for 5 minutes
        "description": "Good balance - allows multiple requests, then frees VRAM"
    },
    "30min": {
        "keep_alive": 1800, # Keep loaded for 30 minutes
        "description": "Good for extended sessions with one model"
    }
}

# GPU Memory Warnings
T4_VRAM_GB = 16
L4_VRAM_GB = 24
A100_VRAM_GB = 40

def get_gpu_info():
    """Get current GPU information"""
    try:
        import subprocess
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', 
                            '--format=csv,noheader,nounits'], 
                          capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            lines = result.stdout.strip().split('\n')
            for line in lines:
                if 'Tesla T4' in line:
                    return {'name': 'T4', 'vram_gb': T4_VRAM_GB}
                elif 'L4' in line or 'RTX A5000' in line:
                    return {'name': 'L4', 'vram_gb': L4_VRAM_GB}
                elif 'A100' in line or 'A100-SXM4' in line:
                    return {'name': 'A100', 'vram_gb': A100_VRAM_GB}
    except:
        pass
    return {'name': 'Unknown', 'vram_gb': T4_VRAM_GB}  # Conservative default

def check_model_memory_safety(model_name, tag):
    """Check if model might exceed GPU memory"""
    # Rough size estimates (in GB)
    model_sizes = {
        '32b': 20, '70b': 40, '405b': 240,  # Very large models
        '14b': 9, '13b': 8, '34b': 20,     # Large models
        '8b': 5, '7b': 4, '6.7b': 3.8,     # Medium models
        '3b': 2, '1b': 1,                     # Small models
    }
    
    # Extract size from tag
    model_size_gb = None
    for size_str, size_gb in model_sizes.items():
        if size_str in tag.lower():
            model_size_gb = size_gb
            break
    
    if model_size_gb is None:
        # Default estimate based on model name
        if '32b' in model_name.lower() or '70b' in model_name.lower():
            model_size_gb = 20
        elif '14b' in model_name.lower() or '13b' in model_name.lower():
            model_size_gb = 9
        else:
            model_size_gb = 7  # Conservative default
    
    gpu_info = get_gpu_info()
    safety_margin = 2  # GB for system overhead
    
    return {
        'model_size_gb': model_size_gb,
        'gpu_vram_gb': gpu_info['vram_gb'],
        'safe': model_size_gb + safety_margin <= gpu_info['vram_gb'],
        'gpu_name': gpu_info['name']
    }

# Create memory management widget
import ipywidgets as widgets
from IPython.display import display, clear_output

memory_style = {'description_width': '120px'}
w_memory = widgets.RadioButtons(
    options=[
        ('Keep loaded (fast, risky)', 'keep_loaded'),
        ('Auto-unload (safe, slow)', 'auto_unload'),
        ('Keep 5 minutes', '5min'),
        ('Keep 30 minutes', '30min')
    ],
    value='keep_loaded',
    description='Memory policy:',
    style=memory_style,
    layout=widgets.Layout(width='100%')
)

w_memory_info = widgets.HTML("")

def update_memory_info(change):
    """Update memory safety information"""
    if 'MODEL_NAME' in globals() and MODEL_NAME:
        model_part, tag_part = MODEL_NAME.split(':', 1) if ':' in MODEL_NAME else (MODEL_NAME, 'latest')
        safety = check_model_memory_safety(model_part, tag_part)
        
        memory_config = MEMORY_OPTIONS[w_memory.value]
        
        if not safety['safe']:
            w_memory_info.value = f"""
            <div style="background-color: #ffebee; padding: 10px; border-radius: 5px; margin: 5px 0;">
                <strong>⚠️ GPU Memory Warning</strong><br>
                Model (~{safety['model_size_gb']}GB) may exceed {safety['gpu_name']} VRAM ({safety['gpu_vram_gb']}GB)<br>
                <strong>Recommendation:</strong> Use "Auto-unload" or smaller model<br>
                Current policy: {memory_config['description']}
            </div>
            """
        elif safety['model_size_gb'] > safety['gpu_vram_gb'] - 4:
            w_memory_info.value = f"""
            <div style="background-color: #fff3e0; padding: 10px; border-radius: 5px; margin: 5px 0;">
                <strong>⚡ High Memory Usage</strong><br>
                Model (~{safety['model_size_gb']}GB) uses most of {safety['gpu_name']} VRAM ({safety['gpu_vram_gb']}GB)<br>
                Policy: {memory_config['description']}
            </div>
            """
        else:
            w_memory_info.value = f"""
            <div style="background-color: #e8f5e8; padding: 10px; border-radius: 5px; margin: 5px 0;">
                <strong>✅ Safe Memory Usage</strong><br>
                Model (~{safety['model_size_gb']}GB) fits comfortably in {safety['gpu_name']} VRAM ({safety['gpu_vram_gb']}GB)<br>
                Policy: {memory_config['description']}
            </div>
            """

w_memory.observe(update_memory_info, names='value')

# Display memory management section
print("🎛️ Memory Management Configuration")
gpu_info = get_gpu_info()
print(f"📍 GPU: {gpu_info['name']} ({gpu_info['vram_gb']}GB VRAM detected)")

display(widgets.VBox([
    widgets.HTML("<h4>🧠 Memory Management</h4>"),
    widgets.HTML("""
    <p><strong>Choose how to handle model memory:</strong></p>
    <ul>
        <li><strong>Keep loaded:</strong> Fastest responses, but may crash with large models</li>
        <li><strong>Auto-unload:</strong> Safest option, model unloaded after each request</li>
        <li><strong>5/30 minutes:</strong> Good balance for multiple requests</li>
    </ul>
    """),
    w_memory,
    w_memory_info
], layout=widgets.Layout(padding='10px', border='1px solid #ddd', border_radius='5px')))

print("💡 Configure memory settings, then continue to tunnel setup below.")
```

### 🌐 Tunnel Configuration

Use the dropdown to choose your tunnel. **Default is Cloudflare quick tunnel** (no credentials needed).

To use a stable URL, add a secret in Colab's key icon sidebar:

| Secret name | Where to get it |
|---|---|
| `cloudflare_token` | [Cloudflare dashboard](https://one.dash.cloudflare.com) → Networks → Tunnels → Create tunnel |
| `ngrok_authtoken` | [ngrok dashboard](https://dashboard.ngrok.com/get-started/your-authtoken) |

```python
# Tunnel selection widget
tunnel_style = {'description_width': '100px'}
w_tunnel = widgets.Dropdown(
    options=[
        ('Cloudflare Quick Tunnel (no credentials)', 'quick'),
        ('Cloudflare Named Tunnel', 'named'),
        ('ngrok', 'ngrok')
    ],
    value='quick',
    description='Tunnel type:',
    style=tunnel_style,
    layout=widgets.Layout(width='100%')
)

display(widgets.VBox([
    widgets.HTML("<h4>🌐 Tunnel Configuration</h4>"),
    w_tunnel
], layout=widgets.Layout(padding='10px', margin='10px 0')))
```

### 📋 Configuration Summary

Your settings will be applied when you run the server cells below.

```python
# Apply memory and tunnel settings
def apply_configuration():
    """Apply the selected memory and tunnel settings"""
    global MEMORY_CONFIG, TUNNEL_TYPE
    
    MEMORY_CONFIG = MEMORY_OPTIONS[w_memory.value]
    TUNNEL_TYPE = w_tunnel.value
    
    print(f"✅ Memory policy: {w_memory.value}")
    print(f"✅ Memory keep_alive: {MEMORY_CONFIG['keep_alive']} seconds")
    print(f"✅ Tunnel type: {TUNNEL_TYPE}")
    
    # Update environment for next cells
    import os
    os.environ['OLLAMA_KEEP_ALIVE'] = str(MEMORY_CONFIG['keep_alive'])
    
    return MEMORY_CONFIG, TUNNEL_TYPE

# Auto-apply when settings change
w_memory.observe(lambda _: apply_configuration(), names='value')
w_tunnel.observe(lambda _: apply_configuration(), names='value')

# Initial application
MEMORY_CONFIG, TUNNEL_TYPE = apply_configuration()
```

In [ ]:
import requests
from bs4 import BeautifulSoup
import ipywidgets as widgets
from IPython.display import display, clear_output

# ── Scrape Ollama library ────────────────────────────────────────────────────
def fetch_ollama_library():
    """Scrape ollama.com/library and return a list of model dicts."""
    print("Fetching model list from ollama.com/library ...")
    resp = requests.get("https://ollama.com/library", timeout=15,
                        headers={"User-Agent": "Mozilla/5.0"})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")

    models = []
    # Attempting more robust selection
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.startswith("/library/") and href.count("/") == 2:
            name = href.split("/")[-1]
            if any(m["name"] == name for m in models): continue
            
            name_el   = a.select_one("h2, [class*='truncate']")
            desc_el   = a.select_one("p, [class*='description']")
            pulls_el  = a.find(string=lambda t: t and "Pulls" in t)
            
            desc = desc_el.get_text(strip=True) if desc_el else ""
            pulls = pulls_el.strip() if pulls_el else ""
            
            tags = [s.get_text(strip=True) for s in a.select("span, li")
                    if s.get_text(strip=True) and len(s.get_text(strip=True)) < 20]
            
            seen = set(); unique_tags = []
            for t in tags:
                if t not in seen and t.lower() != name.lower():
                    seen.add(t); unique_tags.append(t)

            models.append({
                "name": name,
                "desc": desc,
                "pulls": pulls,
                "tags": unique_tags,
            })

    print(f"Found {len(models)} models.")
    return models

ALL_MODELS = fetch_ollama_library()

# ── Derive size tags for each model ──────────────────────────────────────────
SIZE_SUFFIXES = ("b", "m", "k")
def size_tags(model):
    return [t for t in model["tags"]
            if t.lower().replace(".","").rstrip("bmk").isdigit() or
               any(t.lower().endswith(s) for s in SIZE_SUFFIXES)]

# ── Capability categories ─────────────────────────────────────────────────────
CAPABILITIES = ["all", "reasoning", "tools", "vision", "embedding", "code"]
CAP_KEYWORDS  = {
    "reasoning": ["thinking", "reasoning", "r1", "o1"],
    "tools":     ["tools", "function calling"],
    "vision":    ["vision", "multimodal"],
    "embedding": ["embedding", "embed"],
    "code":      ["code", "coder", "coding", "starcoder", "deepseek-coder", "codellama"],
}

def model_matches_cap(model, cap):
    if cap == "all":
        return True
    kws = CAP_KEYWORDS[cap]
    combined = (model["name"] + " " + model["desc"] + " " + " ".join(model["tags"])).lower()
    return any(k in combined for k in kws)

# ── Widgets ──────────────────────────────────────────────────────────────────
_style  = {"description_width": "80px"}
_layout = widgets.Layout(width="100%")

w_search = widgets.Text(
    placeholder="Search models...", description="Search:", style=_style, layout=_layout)
w_cap = widgets.ToggleButtons(
    options=CAPABILITIES, value="all", description="Filter:",
    style={"button_width": "100px", "description_width": "80px"},
    layout=_layout)
w_list = widgets.Select(
    options=[], rows=12, description="Model:", style=_style,
    layout=widgets.Layout(width="100%"))
w_tag_label = widgets.HTML("<b>Tag / variant:</b>")
w_tags = widgets.RadioButtons(options=[], description="", layout=_layout)
w_info  = widgets.HTML("")
w_apply = widgets.Button(description="✅  Use this model",
                         button_style="success",
                         layout=widgets.Layout(width="220px"))
w_out   = widgets.Output()

# ── Tag fetcher ───────────────────────────────────────────────────────────────
_tag_cache = {}

def fetch_tags(model_name):
    if model_name in _tag_cache:
        return _tag_cache[model_name]
    try:
        resp = requests.get(f"https://ollama.com/library/{model_name}/tags",
                            timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        tag_els = soup.select("a[href*='/library/'][href*=':']")
        tags = []
        for el in tag_els:
            tag = el["href"].split(":")[-1].split("/")[0].strip()
            if tag and ":" not in tag and tag not in tags:
                tags.append(tag)
        if not tags:
            tags = ["latest"]
        _tag_cache[model_name] = tags
        return tags
    except Exception:
        _tag_cache[model_name] = ["latest"]
        return ["latest"]

# ── Refresh model list ────────────────────────────────────────────────────────
def refresh_list(*_):
    query = w_search.value.strip().lower()
    cap   = w_cap.value
    filtered = [m for m in ALL_MODELS
                if model_matches_cap(m, cap)
                and (not query or query in m["name"].lower() or query in m["desc"].lower())]
    labels = []
    for m in filtered:
        sizes = [t for t in size_tags(m)][:4]
        size_str = "  " + " ".join(sizes) if sizes else ""
        pulls = m["pulls"].replace(" Pulls", "").strip() if m["pulls"] else ""
        pulls_str = f"  [{pulls}]" if pulls else ""
        labels.append(f"{m['name']}{size_str}{pulls_str}")
    w_list.options = labels
    if labels:
        w_list.value = labels[0]

# ── Update tag radio buttons when model selected ──────────────────────────────
def on_model_select(change):
    if not w_list.value:
        return
    model_name = w_list.value.split()[0]
    w_info.value = "<i>Loading tags...</i>"
    w_tags.options = []
    tags = fetch_tags(model_name)
    ordered = sorted(tags, key=lambda t: (t != "latest", t))[:20]
    w_tags.options = ordered
    if ordered:
        w_tags.value = ordered[0]
    meta = next((m for m in ALL_MODELS if m["name"] == model_name), None)
    desc = meta["desc"] if meta else ""
    pulls = meta["pulls"] if meta else ""
    w_info.value = (
        f"<b>{model_name}</b>"
        + (f" — {desc}" if desc else "")
        + (f"<br><small>{pulls}</small>" if pulls else "")
    )

# ── Apply button ──────────────────────────────────────────────────────────────
MODEL_NAME = None

def on_apply(_):
    global MODEL_NAME
    if not w_list.value or not w_tags.value:
        return
    model_base = w_list.value.split()[0]
    tag        = w_tags.value
    MODEL_NAME = f"{model_base}:{tag}"
    with w_out:
        clear_output()
        print(f"✅  MODEL_NAME set to: {MODEL_NAME}")
        print("Continue to Step 3 when ready.")

w_search.observe(refresh_list, names="value")
w_cap.observe(refresh_list, names="value")
w_list.observe(on_model_select, names="value")
w_apply.on_click(on_apply)

# ── Initial render ────────────────────────────────────────────────────────────
refresh_list()

display(widgets.VBox([
    widgets.HTML("<h4>🦙 Ollama Model Browser</h4>"),
    w_search,
    w_cap,
    w_list,
    w_tag_label,
    w_tags,
    w_info,
    w_apply,
    w_out,
], layout=widgets.Layout(padding="8px", border="1px solid #ccc", border_radius="6px")))

## Step 3: Configure Environment & Memory Management

### Memory Management

Choose how to handle model memory based on your needs and Colab's VRAM limits:

```python
# Memory Management Options
MEMORY_OPTIONS = {
    "keep_loaded": {
        "keep_alive": -1,  # Keep model in VRAM forever (current behavior)
        "description": "Fastest response times, but may cause OOM with large models"
    },
    "auto_unload": {
        "keep_alive": 0,   # Unload model immediately after each request
        "description": "Safe for any model size, but slower response times"
    },
    "5min": {
        "keep_alive": 300,  # Keep loaded for 5 minutes
        "description": "Good balance - allows multiple requests, then frees VRAM"
    },
    "30min": {
        "keep_alive": 1800, # Keep loaded for 30 minutes
        "description": "Good for extended sessions with one model"
    }
}

# GPU Memory Warnings
T4_VRAM_GB = 16
L4_VRAM_GB = 24
A100_VRAM_GB = 40

def get_gpu_info():
    """Get current GPU information"""
    try:
        import subprocess
        result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', 
                            '--format=csv,noheader,nounits'], 
                          capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            lines = result.stdout.strip().split('\n')
            for line in lines:
                if 'Tesla T4' in line:
                    return {'name': 'T4', 'vram_gb': T4_VRAM_GB}
                elif 'L4' in line or 'RTX A5000' in line:
                    return {'name': 'L4', 'vram_gb': L4_VRAM_GB}
                elif 'A100' in line or 'A100-SXM4' in line:
                    return {'name': 'A100', 'vram_gb': A100_VRAM_GB}
    except:
        pass
    return {'name': 'Unknown', 'vram_gb': T4_VRAM_GB}  # Conservative default

def check_model_memory_safety(model_name, tag):
    """Check if model might exceed GPU memory"""
    # Rough size estimates (in GB)
    model_sizes = {
        '32b': 20, '70b': 40, '405b': 240,  # Very large models
        '14b': 9, '13b': 8, '34b': 20,     # Large models
        '8b': 5, '7b': 4, '6.7b': 3.8,     # Medium models
        '3b': 2, '1b': 1,                     # Small models
    }
    
    # Extract size from tag
    model_size_gb = None
    for size_str, size_gb in model_sizes.items():
        if size_str in tag.lower():
            model_size_gb = size_gb
            break
    
    if model_size_gb is None:
        # Default estimate based on model name
        if '32b' in model_name.lower() or '70b' in model_name.lower():
            model_size_gb = 20
        elif '14b' in model_name.lower() or '13b' in model_name.lower():
            model_size_gb = 9
        else:
            model_size_gb = 7  # Conservative default
    
    gpu_info = get_gpu_info()
    safety_margin = 2  # GB for system overhead
    
    return {
        'model_size_gb': model_size_gb,
        'gpu_vram_gb': gpu_info['vram_gb'],
        'safe': model_size_gb + safety_margin <= gpu_info['vram_gb'],
        'gpu_name': gpu_info['name']
    }

# Create memory management widget
import ipywidgets as widgets
from IPython.display import display, clear_output

memory_style = {'description_width': '120px'}
w_memory = widgets.RadioButtons(
    options=[
        ('Keep loaded (fast, risky)', 'keep_loaded'),
        ('Auto-unload (safe, slow)', 'auto_unload'),
        ('Keep 5 minutes', '5min'),
        ('Keep 30 minutes', '30min')
    ],
    value='keep_loaded',
    description='Memory policy:',
    style=memory_style,
    layout=widgets.Layout(width='100%')
)

w_memory_info = widgets.HTML("")

def update_memory_info(change):
    """Update memory safety information"""
    if 'MODEL_NAME' in globals() and MODEL_NAME:
        model_part, tag_part = MODEL_NAME.split(':', 1) if ':' in MODEL_NAME else (MODEL_NAME, 'latest')
        safety = check_model_memory_safety(model_part, tag_part)
        
        memory_config = MEMORY_OPTIONS[w_memory.value]
        
        if not safety['safe']:
            warning_html = (
                '<div style="background-color: #ffebee; padding: 10px; border-radius: 5px; margin: 5px 0;">'
                '<strong>WARNING: GPU Memory Warning</strong><br>'
                f'Model (~{safety["model_size_gb"]}GB) may exceed {safety["gpu_name"]} VRAM '
                f'({safety["gpu_vram_gb"]}GB).<br>'
                '<strong>Recommendation:</strong> Use Auto-unload or smaller model.<br>'
                f'Current policy: {memory_config["description"]}</div>'
            )
            w_memory_info.value = warning_html
        elif safety['model_size_gb'] > safety['gpu_vram_gb'] - 4:
            warning_html = (
                '<div style="background-color: #fff3e0; padding: 10px; border-radius: 5px; margin: 5px 0;">'
                '<strong>High Memory Usage</strong><br>'
                f'Model (~{safety["model_size_gb"]}GB) uses most of {safety["gpu_name"]} VRAM '
                f'({safety["gpu_vram_gb"]}GB).<br>'
                f'Policy: {memory_config["description"]}</div>'
            )
            w_memory_info.value = warning_html
        else:
            safe_html = (
                '<div style="background-color: #e8f5e8; padding: 10px; border-radius: 5px; margin: 5px 0;">'
                '<strong>Safe Memory Usage</strong><br>'
                f'Model (~{safety["model_size_gb"]}GB) fits comfortably in {safety["gpu_name"]} VRAM '
                f'({safety["gpu_vram_gb"]}GB).<br>'
                f'Policy: {memory_config["description"]}</div>'
            )
            w_memory_info.value = safe_html

w_memory.observe(update_memory_info, names='value')

# Display memory management section
print("Memory Management Configuration")
gpu_info = get_gpu_info()
print(f"GPU: {gpu_info['name']} ({gpu_info['vram_gb']}GB VRAM detected)")

display(widgets.VBox([
    widgets.HTML("<h4>Memory Management</h4>"),
    widgets.HTML("""
    <p><strong>Choose how to handle model memory:</strong></p>
    <ul>
        <li><strong>Keep loaded:</strong> Fastest responses, but may crash with large models</li>
        <li><strong>Auto-unload:</strong> Safest option, model unloaded after each request</li>
        <li><strong>5/30 minutes:</strong> Good balance for multiple requests</li>
    </ul>
    """),
    w_memory,
    w_memory_info
], layout=widgets.Layout(padding='10px', border='1px solid #ddd', border_radius='5px'))

In [ ]:
# @title Tunnel configuration { run: "auto" }
TUNNEL_METHOD = "Cloudflare quick tunnel (no credentials)"  # @param ["Cloudflare quick tunnel (no credentials)", "Cloudflare named tunnel (token)", "ngrok (token)"]

import os
import re
import json
import subprocess
import threading
import time
import requests
from google.colab import userdata

os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia'

# ── Verify model was selected in Step 2 ─────────────────────────────────────
if not 'MODEL_NAME' in globals() or not MODEL_NAME:
    raise RuntimeError(
        "MODEL_NAME is not set.\n"
        "Go back to Step 2, choose a model and tag, then click '✅ Use this model'."
    )

# ── Environment variables ────────────────────────────────────────────────────
os.environ['OLLAMA_KEEP_ALIVE']      = '-1'
os.environ['OLLAMA_CONTEXT_LENGTH']  = '16384'
os.environ['OLLAMA_HOST']            = '0.0.0.0'

# ── Map dropdown to internal tunnel type ─────────────────────────────────────
_method_map = {
    "Cloudflare quick tunnel (no credentials)": "quick",
    "Cloudflare named tunnel (token)":          "cloudflare",
    "ngrok (token)":                            "ngrok",
}
tunnel_type  = _method_map[TUNNEL_METHOD]
tunnel_token = None

if tunnel_type == "cloudflare":
    try:
        tunnel_token = userdata.get("cloudflare_token")
        if not tunnel_token:
            raise ValueError("Secret 'cloudflare_token' is empty.")
        print("Cloudflare named tunnel — token loaded.")
    except Exception as e:
        print(f"ERROR: {e}")
        print("Add 'cloudflare_token' to Colab Secrets, or switch to the quick tunnel option.")
        raise

elif tunnel_type == "ngrok":
    try:
        tunnel_token = userdata.get("ngrok_authtoken")
        if not tunnel_token:
            raise ValueError("Secret 'ngrok_authtoken' is empty.")
        print("ngrok — token loaded.")
        print("NOTE: ngrok free-tier URLs rotate every ~2 hours.")
    except Exception as e:
        print(f"ERROR: {e}")
        print("Add 'ngrok_authtoken' to Colab Secrets, or switch to the quick tunnel option.")
        raise
else:
    print("Cloudflare quick tunnel — no credentials needed.")
    print("NOTE: The public URL will change every time you run Step 6.")

# ── Process registry ─────────────────────────────────────────────────────────
_bg_processes = {}

print(f"\nModel  : {MODEL_NAME}")
print(f"Tunnel : {TUNNEL_METHOD}")

## Step 4: Helper Functions

In [ ]:
def stream_output(process, label=""):
    """Read process stdout in a background thread and print each line."""
    prefix = f"[{label}] " if label else ""
    for line in process.stdout:
        line = line.strip()
        if line:
            print(f"{prefix}{line}")


def wait_for_ollama(timeout=60):
    """Block until the Ollama HTTP API responds or timeout is reached."""
    for i in range(timeout):
        if _bg_processes.get('ollama') and _bg_processes['ollama'].poll() is not None:
            raise RuntimeError(
                f"ollama serve exited early "
                f"(code {_bg_processes['ollama'].returncode}). "
                "Check the output above — there may be a GPU driver issue."
            )
        try:
            r = requests.get("http://localhost:11434", timeout=2)
            if r.status_code in (200, 404):
                print(f"Ollama is up (after {i+1}s).")
                return
        except requests.exceptions.ConnectionError:
            pass
        time.sleep(1)
    raise RuntimeError("Ollama did not start within 60 s.")


print("Helper functions ready.")

## Step 5: Start Ollama Server

In [ ]:
## Security Advisory

> **CRITICAL SECURITY WARNING** 

Your Ollama instance is now publicly accessible without authentication. In 2026, this creates significant security risks:

### Immediate Risks

- **Unauthorized GPU Usage**: Anyone can use your Colab GPU quota for inference
- **Cost Abuse**: Malicious actors can run expensive operations at your expense
- **Prompt Injection**: Vulnerable to adversarial prompt attacks
- **Tool Exploitation**: Models with tool access can trigger unauthorized actions
- **Data Exfiltration**: Internet-connected models could be manipulated to extract data

### Security Recommendations

#### **Essential for Public Deployment:**
```python
# Add authentication before exposing publicly
# Option 1: API Key Authentication
import os
os.environ['OLLAMA_API_KEY'] = 'your-secure-api-key-here'

# Option 2: Restrict to specific IPs (if supported)
# Add IP whitelist to your tunnel service
```

#### **Operational Security:**
- [YES] **Monitor Usage**: Check Colab GPU usage regularly
- [YES] **Short Sessions**: Use for testing only, not permanent deployment  
- [YES] **Network Safety**: Run only on trusted networks
- [YES] **Tool Control**: Disable tool access for public deployments
- [YES] **Cost Alerts**: Set up billing alerts if possible

#### **Advanced Protection:**
```python
# Example: Add request rate limiting
# This would require a reverse proxy with authentication
# Consider nginx, cloudflare workers, or API gateway
```

### Immediate Actions Required

1. **Add Authentication**: Implement API keys or OAuth
2. **Monitor Costs**: Check Colab usage every few hours
3. **Limit Exposure**: Use only for short testing periods
4. **Secure Tools**: Disable tool access for public endpoints
5. **Network Isolation**: Use VPN or trusted networks only

### If Compromised

If you suspect unauthorized access:
```python
# Emergency shutdown
for name, proc in _bg_processes.items():
    try:
        proc.terminate()
        print(f"Emergency terminated: {name}")
    except:
        pass

# Change all tokens/secrets immediately
# Check Colab usage logs
# Consider rotating tunnel URLs
```

**Remember**: This notebook is for educational purposes. For production use, implement proper authentication and monitoring.

---

## Additional Resources

- [Ollama Security Best Practices](https://github.com/ollama/ollama/blob/main/docs/security.md)
- [Colab Security Guide](https://research.google.com/colab/security)
- [API Security Checklist](https://owasp.org/www-project-api-security/)

## Step 6: Pull Model

In [ ]:
print(f"Pulling {MODEL_NAME} ...")
print("This may take 5–15 minutes depending on your connection speed.")

pull = subprocess.Popen(
    ['ollama', 'pull', MODEL_NAME],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
)
for line in pull.stdout:
    print(line.strip())

rc = pull.wait()
if rc == 0:
    print(f"\nSuccessfully pulled {MODEL_NAME}!")
else:
    raise RuntimeError(f"Model pull failed (exit code {rc}). Try restarting the runtime.")

## Step 7: Start Tunnel

In [ ]:
public_url = None

# ── Cloudflare named tunnel ──────────────────────────────────────────────────
if tunnel_type == 'cloudflare':
    print("Starting Cloudflare named tunnel...")
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--edge-ip-version', '4', '--no-autoupdate', 'run', '--token', tunnel_token],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
    )
    _bg_processes['tunnel'] = cf_proc
    threading.Thread(target=stream_output, args=(cf_proc, 'cf'), daemon=True).start()

    print("Waiting for tunnel to connect (up to 30 s)...")
    for _ in range(15):
        time.sleep(2)
        if cf_proc.poll() is not None:
            raise RuntimeError(
                f"cloudflared exited early (code {cf_proc.returncode}). "
                "Check that your tunnel token is valid and the tunnel is active "
                "in the Cloudflare dashboard."
            )
        try:
            requests.get("http://localhost:11434/api/tags", timeout=3)
            print("Cloudflare Tunnel daemon is running.")
            print("Use the hostname configured in your Cloudflare dashboard as the public URL.")
            public_url = "https://<your-cloudflare-hostname>"
            break
        except Exception:
            pass

# ── Cloudflare quick tunnel ──────────────────────────────────────────────────
elif tunnel_type == 'quick':
    print("Starting Cloudflare quick tunnel (no credentials)...")
    cf_proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--edge-ip-version', '4', '--url', 'http://localhost:11434', '--no-autoupdate'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        universal_newlines=True,
    )
    _bg_processes['tunnel'] = cf_proc

    # Read lines until we find the trycloudflare.com URL, then hand off to a
    # background thread so the cell completes cleanly.
    print("Waiting for public URL...")
    for line in cf_proc.stdout:
        line = line.strip()
        if line:
            print(f"[cf] {line}")
        match = re.search(r'(https://[\w-]+\.trycloudflare\.com)', line)
        if match:
            public_url = match.group(1)
            break

    # Hand remaining stdout to background thread so cell doesn't block
    threading.Thread(target=stream_output, args=(cf_proc, 'cf'), daemon=True).start()

    if not public_url:
        raise RuntimeError("Could not find a Cloudflare quick-tunnel URL in output.")

# ── ngrok ────────────────────────────────────────────────────────────────────
elif tunnel_type == 'ngrok':
    from pyngrok import ngrok as _ngrok
    _ngrok.set_auth_token(tunnel_token)
    _ngrok.kill()
    time.sleep(1)

    print("Starting ngrok tunnel...")
    tunnel = _ngrok.connect(11434, "http", options={"host_header": "localhost:11434"})
    public_url = tunnel.public_url
    print(f"ngrok tunnel open: {public_url}")

# ── Summary ──────────────────────────────────────────────────────────────────
if public_url:
    print('\n' + '='*60)
    print(f'  Ollama is live!')
    print(f'  Model  : {MODEL_NAME}')
    print(f'  URL    : {public_url}')
    print(f'  Tunnel : {tunnel_type}')
    print('='*60)
    if tunnel_type == 'quick':
        print('\nNOTE: This URL is temporary — it changes every time you run this cell.')
        print('Add a cloudflare_token or ngrok_authtoken secret for a stable URL.')
    elif tunnel_type == 'ngrok':
        print('\nNOTE: ngrok free-tier URLs rotate every ~2 hours.')
else:
    print("\nPublic URL could not be determined. Check the output above.")

## Step 8: Test Endpoint

In [ ]:
# @title { run: "auto" }
TEST_PROMPT = "Write a hello world program in Python with a comment on each line."  # @param {type:"string"}

if not public_url or public_url.startswith("https://<your"):
    print("Set public_url to your actual tunnel hostname before running this cell.")
else:
    # Confirm endpoint is reachable
    try:
        r = requests.get(f"{public_url}/api/tags", timeout=10)
        models = [m['name'] for m in r.json().get('models', [])]
        print(f"Endpoint reachable. Available models: {models}")
    except Exception as e:
        print(f"Endpoint not reachable yet: {e}")
        print("Wait a few seconds and re-run this cell.")
    else:
        print(f"\nSending test prompt to {MODEL_NAME}...\n")
        try:
            resp = requests.post(
                f"{public_url}/api/chat",
                json={
                    "model":    MODEL_NAME,
                    "messages": [{"role": "user", "content": TEST_PROMPT}],
                    "stream":   False,
                },
                timeout=120,
            )
            resp.raise_for_status()
            print(resp.json()["message"]["content"])
        except Exception as e:
            print(f"Chat request failed: {e}")

## Step 9: Keep Server Running

Run this cell to keep the server alive. It blocks until you stop it (■) or the session disconnects.

Stopping this cell will **terminate Ollama and the tunnel**.

In [ ]:
if not public_url:
    print("Server not started. Run previous cells first.")
else:
    print(f"Server running at : {public_url}")
    print(f"Model             : {MODEL_NAME}")
    print(f"Tunnel            : {tunnel_type}")
    print(f"Context window    : 16384 tokens")
    print("\nKeeping server alive. Interrupt this cell (■) to shut down.")

    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        print("\nShutting down...")
        for name, proc in _bg_processes.items():
            try:
                proc.terminate()
                print(f"  Terminated {name}")
            except Exception:
                pass
        print("Done.")

---

## Usage Examples

Replace `https://your-url` with the URL printed in Step 7.

### curl

```bash
# Chat
curl -X POST https://your-url/api/chat \
  -H 'Content-Type: application/json' \
  -d '{"model": "qwen2.5:14b", "messages": [{"role": "user", "content": "Explain recursion"}]}'

# Generate (single-turn)
curl -X POST https://your-url/api/generate \
  -H 'Content-Type: application/json' \
  -d '{"model": "qwen2.5:14b", "prompt": "Hello, world!"}'

# Reasoning (DeepSeek R1)
curl -X POST https://your-url/api/chat \
  -H 'Content-Type: application/json' \
  -d '{"model": "deepseek-r1:14b", "messages": [{"role": "user", "content": "Solve step by step: ..."}]}'
```

### Python

```python
import requests

response = requests.post(
    "https://your-url/api/chat",
    json={
        "model": "qwen2.5:14b",
        "messages": [{"role": "user", "content": "Write a Flask API"}],
    }
)
print(response.json()["message"]["content"])
```

### VS Code (Continue / CodeGPT)

1. Install the extension
2. Set provider to **Ollama**
3. Set base URL to your tunnel URL
4. Select model (e.g. `qwen2.5:14b`)

---

## Notes

- **Quick tunnel**: URL changes every run — not suitable for persistent integrations.
- **ngrok free tier**: URL rotates every ~2 hours.
- **Cloudflare named tunnel**: Fixed hostname configured in your Cloudflare dashboard — best for long-running use.
- **Colab free tier**: Sessions last a few hours; daily GPU quota applies.
- **Colab Pro**: Sessions up to 24 hours with L4 GPU access.
- **Security**: The endpoint is public with no authentication. Avoid sending sensitive data.